Ноутбук служит основой для изучения, однако многое нужно будет поискать и почитать самостоятельно
***

# Часть 1: теория (5 pt)

### Данные

В реальной практике **80% времени уходит на анализ и подготовку данных**, а не на построение моделей или визуализацию. Грязные данные ведут к ошибочным выводам:

| Проблема | Последствие для анализа |
|----------|------------------------|
| **Пропуски** | Смещение средних значений, потеря наблюдений |
| **Дубликаты** | Завышенные частоты, искажённая статистика |
| **Аномалии** | Искажение корреляций, неработающие модели МО |
| **Несогласованный текст** | Невозможность группировки по категориям |

В этой части поработаем именно с этими четырьмя пунктами. Однако во второй части вы можете использовать любые известные вам методы.

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

# Вот так можно смотреть версию (некоторые методы зависят от версии)
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.3.3


Мы начнём с [датасета](https://www.kaggle.com/datasets/zynicide/wine-reviews?spm=a2ty_o01.29997173.0.0.17c95171dBJybJ) — **Wine Reviews**. Этот датасет как раз подойдет нам для обсуждения методов.
> В описании датасета можно прочитать описание столбцов и их значение.

In [2]:
wine = pd.read_csv('winemag-data-130k-v2.csv')

print(wine.shape)
print(wine.columns.tolist())

(129971, 14)
['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']


In [3]:
wine.head(3)

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm


Видно, что датасет считался некорректно и у нас появился столбец `Unnamed: 0`. Это произошло потому что pandas автоматически присваивает индексы, но в нашем датасете первый столбец уже был индексами. Это можно исправить с помощью чтения с параметром
```python
wine = pd.read_csv('winemag-data-130k-v2.csv', index_col=0)
```

Но мы пойдем другим способом и просто **удалим** этот столбец с помощью метода [.drop()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html)

In [4]:
wine = wine.drop(columns=['Unnamed: 0'])

Также для того, чтобы нам было проще работать в этом задании удалим еще некоторые столбцы. То есть **оставим** только те, которые нам интересны.

In [5]:
wine = wine[['country', 'description', 'winery', 'designation', 'points', 'price', 'province', 'title', 'variety']]

In [6]:
wine.head()

,country,description,winery,designation,points,price,province,title,variety
0,Italy,"Aromas include tropical fruit, broom, brimston...",Nicosia,Vulkà Bianco,87,NaN,Sicily & Sardinia,Nicosia 2013 Vulkà Bianco (Etna),White Blend
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Quinta dos Avidagos,Avidagos,87,15.0,Douro,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red
2,US,"Tart and snappy, the flavors of lime flesh and...",Rainstorm,NaN,87,14.0,Oregon,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris
3,US,"Pineapple rind, lemon pith and orange blossom ...",St. Julian,Reserve Late Harvest,87,13.0,Michigan,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling
4,US,"Much like the regular bottling from 2012, this...",Sweet Cheeks,Vintner's Reserve Wild Child Block,87,65.0,Oregon,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir


### Диагностика проблем (1 pt)

Перед любыми действиями нужно **понять, какие проблемы есть в данных**. Базовый чек-лист:

| Проблема | Метод диагностики | Что ищем |
|----------|-------------------|----------|
| **Пропуски** | `.isna().sum()` | Количество `NaN` по столбцам |
| **Дубликаты** | `.duplicated().sum()` | Полные или частичные повторы строк |
| **Аномалии** | `.describe()` + визуализация | Значения за пределами ожидаемого диапазона |
| **Несогласованный текст** | `.value_counts()` | Опечатки, разный регистр, лишние пробелы |

Сууждения об аномалиях можно делать на основе жизненной (бизнес) логики (не может кот весить тонну) и/или странных сочетаний `min`, `mean`, `max` и других статистик.  
В репорте `value_counts` видно, сколько раз повторяется значение. Поэтому если строковое значение повторяется только один раз (хотя при это, например, категория), то стоит задуматься о том, что нужно данные как-то преобразовать. 

> Про некоторые методы вы уже знаете, а про остальные можно посмотреть в документации: [pandas.DataFrame.isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html), [pandas.DataFrame.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html).

***

Предположим, что мы совсем не разбираемся в вине (надеюсь, что вам и предполагать не надо) и хотим что-то про него понять. Например какое и за сколько лучше всего.

In [ ]:
# ЗАДАНИЕ:
#          1. Посчитайте количество пропусков (NaN) по каждому столбцу
#          2. Найдите количество полных дубликатов строк
#          3. Выведите базовую статистику для числовых столбцов
#          4. Проанализируйте распределение цены — есть ли подозрительные значения?
#          5. Напишите небольшой вывод о водможных проблемах с данными

...

In [ ]:
# Поменяйте тип ячейки на Marcdown, удалите комментарий и напишите вывод
**ВЫВОД:**

<details>
  <summary>Когда будете готовы, кликните сюда, чтобы посмотреть краткую сводку</summary>
    <ol>
      <li> Безусловно для нас важнее всего цена, а в этом столбце много пропусков, нужно с ними что-то делать. Также в цене есть явно аномальные (сильно маленькие или сильно большие значения), нужно понять, как с ними работать.</li>
      <li>В целом в разных столбцах есть пропуски, но вот в столбце designation их слишком много (37к).</li>
      <li>Рейтинги не содержат явных аномалий — все значения в ожидаемом диапазоне (80-100 баллов).</li>
    </ol> 
</details>

### Обработка дубликатов (0,5 pt)

In [7]:
wine.duplicated().sum()

np.int64(9983)

Дубликаты — это полные или частичные повторы строк, которые искажают статистику. В датасете вин есть 9 983 дубликата. Такие повторы будут портить нам статистику, поэтому их лучше удалить.
Поможет в этом [.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [ ]:
# ЗАДАНИЕ: Удалите дубликаты, оставив только первую запись
wine = ...

In [ ]:
wine.duplicated().sum()

### Обработка пропусков (1 pt)

Выбор стратегии работы с пропусками зависит от **природы пропуска**. Например для цены нам критично заполнение пропусков, а вот в столбце `designation` (виноградник на территории винодельни, откуда собирают виноград, из которого делают вино) пропуски для нас некритичны и мы можем заменить их на нейтральное значение нужного типа (`str` "Undefined").

Прочитать про то, как можно работать с пропусками можно много где, например, [тут](https://www.geeksforgeeks.org/data-analysis/working-with-missing-data-in-pandas/) и [здесь](https://www.geeksforgeeks.org/data-analysis/handling-missing-values-machine-learning/).

Главное при заполнении пропусков следовать логике и делать так, чтобы стратегия заполнения не ломала данные. Например можно заполнять пропуски **средним (или наиболее частым) значением по какому-то условию**. Если пропусков мало или они не значимы, то можно удалять такие строки.

_Иногда важно смотреть внимательно: не всегда пропуски видны, иногда вместо `NaN` значения содержат нули или неопределенные слова. Такие случаи тоже можно считать пропусками, но в этом датасете такого нет._

***

Сейчас мы не будем сильно заморачиваться с этим и просто попробуем разные методы:
* в столбце `country` **заменим пропуски на наиболее часто встречаемую** страну (пропусков не так много, чтобы они повлияли, да и если страна неизвестна, то с оцень большой вероятностью это Итария или Франция)
* В столбце `designation` заменим пропуски на метку "Undefined". Это нормально, что для массовых вин никого не заботит виноградник. Но для того, чтобы мы корректно могли работать со строками заменим наны на строку.
* В столбце `price` заменим пропуски **медианой** по стране. Медианой, а не средним для того, чтобы не влияли выбросы и не появлялись дробные значения.
* В столбце `province` заполним пропуски **модой** по стране (наиболее частым).
* Строку с пропуском в столбце `variety` просто **удалим**)

Для начала сделаем **копию датасета** для того, чтобы не потерять изначальные данные!

Вам помогут: [.fillna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html), [.transform()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html), [.dropna()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html).

In [8]:
wine.isna().sum()

country           63
description        0
winery             0
designation    37465
points             0
price           8996
province          63
title              0
variety            1
dtype: int64

In [ ]:
wine_clean = wine.copy()

In [ ]:
# ЗАДАНИЕ: обработайте пропуски согласно пунктам выше

most_common_country = ...
wine_clean['country'] = ...

wine_clean['designation'] = ...

country_median_price = ...
wine_clean['price'] = ...

country_mode_province = ...
wine_clean['province'] = ...

wine_clean = wine_clean.dropna(subset=...)

In [ ]:
# Должно быть по нулям
wine_clean.isna().sum()

### Обработка аномалий (0,5 pt)

Аномалии (выбросы) — это значения, которые **значительно отклоняются** от остальных наблюдений. Они могут быть:
- **Ошибка измерения/ввода** (цена вина отрицательна) -> удалить
- **Редкое, но валидное наблюдение** (винтажное вино за много денег) -> сохранить или обработать отдельно
- **Систематическое искажение** (группировка значений на границах) -> исследовать источник

_Можно выделить три подхода для обнаружения аномалий или подбора порогов_

| Подход | Как работает | Плюсы | Минусы | Когда применять |
|--------|--------------|-------|--------|-----------------|
| **Бизнес-логика** | Определение границ на основе экспертных знаний предметной области | Интерпретируемо, учитывает контекст, не требует сложных вычислений | Требует экспертизы, субъективно | Когда есть чёткие физические/экономические ограничения (цена ≥ 0, возраст 0–120 лет) |
| **Статистика (IQR)** | Границы: `Q1 - 1.5×IQR` и `Q3 + 1.5×IQR`, где `IQR = Q3 - Q1` | Автоматизируемо, не зависит от распределения | Может удалить валидные экстремумы, чувствителен к выбору коэффициента (1.5 vs 3.0) | Когда нет экспертных знаний, но нужно быстро найти «подозрительные» значения |
| **Статистика (Z-score)** | Границы: $|z| > 3$, где $z = (x - \mu) / \sigma$ | Хорошо работает для нормальных распределений | Требует нормальности данных, чувствителен к выбросам при расчёте $\mu$ и $\sigma$ | Для признаков с приблизительно нормальным распределением |

Вот тут есть [гайд](https://sky.pro/wiki/media/kakie-sushhestvuyut-metody-opredeleniya-vybrosov-v-dannyh/) про то, как определять выбросы.

> Также для этого поможет документация: [.quantile()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html), [scipy.stats.zscore](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.zscore.html).

**Что же делать с аномалиями?**

| Стратегия | Когда применять | Пример |
|----------|-------------------|----------|
| **Удаление** | Аномалия — явная ошибка (цена $\leqslant$ $0) | `df = df[df['price'] > 0]` |
| **Ограничение (capping)** | Аномалия — экстремум, но не ошибка | `df['price'] = df['price'].clip(upper=500)` |
| **Сегментация** | Аномалии образуют отдельный сегмент | `df['segment'] = pd.cut(df['price'], bins=[0, 50, 100, float('inf')])` |
| **Сохранение с пометкой** | Нужно анализировать аномалии отдельно | `df['is_outlier'] = df['price'] > 500` |

Кажется, что в нашем случае лучшим решением будет **поделить** вина **на категории**: дешевые, нормальные, очень дорогие. Для того, чтобы лучше понять данные посмотрим на **диаграмму** цен ([DataFrame.plot.hist](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.hist.html))

In [ ]:
wine_clean.price.plot.hist()

_Какие пороги для ценовых категорий стоит поставить?_

In [ ]:
# ЗАДАНИЕ: 1. Создать новый столбец wine_clean['price_category']
#          2. Согласно выявленным порогам определить ценовую категорию каждого вина

wine_clean['price_category'] = ...

In [ ]:
wine_clean['price_category'].value_counts()

### Обработка текста (0,5 pt)

Текстовые данные в реальных датасетах часто содержат **несогласованные обозначения**, которые мешают корректной группировке и анализу. Например могут быть следующие проблемы:

| Столбец | Проблема | Примеры из данных |
|---------|----------|-------------------|
| `country` | Разные форматы для одной страны | `'US'`, `'USA'`, `'U.S.A.'`, `'United States'` |
| `province` | Разный регистр и лишние пробелы | `' california '`, `'California'`, `'CALIFORNIA'` |
| `variety` | Опечатки и синонимы | `'Pinot Noir'` vs `'PinotNoir'` |

Нам повезло, в нашем датасете такого нет, однако, встретиться такое может часто.

**Как приводить текстовые/категориальные данные в общий вид?**

| Метод | Назначение | Пример |
|-------|------------|--------|
| `.str.strip()` | Удаляет пробелы по краям | `' California '` → `'California'` |
| `.str.lower()` / `.str.title()` | Приводит к единому регистру | `'california'` → `'California'` |
| `.replace()` | Заменяет значения по словарю | `{'US': 'United States'}` |
| `.str.contains()` | Поиск подстроки (регистронезависимый) | `contains('berry', case=False)` |

Как всегда подробнее в документации: [.str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html),
[.str.lower()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.lower.html),
[.replace()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.replace.html),
[.str.contains()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html).

In [ ]:
# Проверьте, что в нашем датасете нет таких проблем
...

Что касается текстовых данных, то у нас есть очень информативный столбец `description`, из которого можно вытащить много информации. Для того, чтобы это было сделать легче, приведем все буквы к строчному виду и удалим знаки препинания.

In [ ]:
# ЗАДАНИЕ: приведите описания к нижнему регистру и удалите знаки препинания

wine_clean['description'] = ...

In [ ]:
wine_clean.description[42]

### Вывод (1,5 pt)

Теперь по подготовленным данным ответьте на следующие вопросы (**напишите код и небольшие выводы**):
- Какая страна производит самые дорогие вина (по средней цене)? Какая — самые дешёвые? Какие страны производят очень дорогие вина?
- Есть ли связь между ценой и рейтингом? Вина какой ценовой категории получают самый высокий средний рейтинг?
- Какие 3 сорта винограда (variety) самые лучшие (по цене, по рейтингу)?
- Сравните средний рейтинг разных ценовых сегментов. Насколько отличается качество между бюджетным и премиум-сегментом?
- В каких ценовых сегментах встречаются фруктовые вина (посмотрите вхождения ключевых слов в описания)? 

In [ ]:
...

...

# Часть 2: практика (5 pt)

**В этой части вам нужно самостоятельно провести анализ датасета и сделать выводы.**
***
**ЗАДАНИЕ: подготовить данные и сделать три неочевидных/интересных вывода (основывающихся на данных)**
***
В качестве датасета возьмем данные [Netflix](https://www.kaggle.com/datasets/shivamb/netflix-shows/data). 

> Оцениваться будут следующие пункты:
> * Диагностика проблем (1 pt)
> * Подготовка и очистка данных, в том числе добавление новых столбцов, выделение категорий и тд. Важна аргуменитация и пояснения выбора методов.(2 pt)
> * Анализ данных и выводы (2 pt)

В качестве отправной точки можете использовать вопросы:
- Какие 3 страны производят больше всего контента для Netflix? Есть ли различия между фильмами и сериалами?
- В каком году было выпущено максимальное количество сериалов?
- Какой возрастной рейтинг самый распространённый для фильмов? Для сериалов? Есть ли страна, где доминируют контент со взрослым рейтингом?
- Какова медианная продолжительность фильмов (в минутах)? Сериалов (в сезонах)?
- Какой жанр доминирует в США? В Индии?
- Какой тип контента «старше»?

**Пожалуйста, оформляйте ваш код и выводы аккуратно, при необходимости добавляйте секции и форматируйте текст.**

In [ ]:
movie = pd.read_csv('netflix_titles.csv')
movie.shape

In [ ]:
movie.head()

In [ ]:
...

**УДОСТОВЕРЬТЕСЬ, ЧТО ВСЕ ВАШИ ВЫВОДЫ И КОД ОФОРМЛЕНЫ ТАК, ЧТОБЫ ЛЕГКО МОЖНО БЫЛО ПОНЯТЬ, К ЧЕМУ ОНИ ОТНОСЯТСЯ**  
**НЕ ЗАБУДЬТЕ ПЕРЕЗАПУСТИТЬ НОУТБУК ТАК, ЧТОБЫ ВСЕ ЯЧЕЙКИ БЫЛИ ЗАПУЩЕНЫ ПОДРЯД, НАЧИНАЯ С [1]**